n2p2 dataset might look like,

```python
begin                                       --- begining of one atomic configuration
lattice 5.0 0.0 0.0
lattice 0.0 5.0 0.0
lattice 0.0 0.0 5.0
atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0
atom 2.5 2.5 2.5 Si 0.0 0.0 0.0 0.0 0.0
energy -10.5
charge 0.0
end                                         --- end of one atomic configuration
```

Task -- getting atomic structures as ASE objects and energy values separately

In [7]:
from pathlib import Path # class 'Path' that helps locating the files
from ase import Atoms # class 'Atoms' represents atomic structures
import numpy as np
from dscribe.descriptors import ACSF

'Atom Object' is like,

```python
atoms = Atoms(
    symbols=["Fe", "Fe"],
    positions=[[0, 0, 0], [2, 2, 2]]
)
```

They can store elements, coordinates, unit cell dimensions, periodic boundry conditions

# TASK 1 - Extract data from the data file 

Internal Helper Functions 

+ not called directly by the user
+ supports the main function

_ at the beginning indicates that this is intended to be an internal helper function

In [ ]:
def _parse_n2p2_block(block): # Internal Helper function to parse a single n2p2 block and extract atomic structure and energy information from the lines in n2p2 file
    symbols = []                # block - lines belongs to one atomic configuration
    positions = []
    cell = []
    energy = None
    has_cell = False

    for line in block:
        parts = line.split() # separate the strings in a line based on whitespace

        if not parts:
            continue # if the list parts is empty, skip to next line

        if parts[0] == "lattice":
            cell.append([float(x) for x in parts[1:4]]) # Make the vector with direction values of [x,y,z] ex - from "lattice 0.0 5.0 0.0" extract [0.0, 5.0, 0.0]
            has_cell = True # lattice information for this block is found successfully

        elif parts[0] == "atom":
            positions.append([float(x) for x in parts[1:4]]) # poistion of the atom to the list positions, ex - from "atom 0.0 0.0 0.0 H" extract [0.0, 0.0, 0.0]
            symbols.append(parts[4]) # Type of atom to the list symbols, ex - from "atom 0.0 0.0 0.0 Si" extract "Si"
                                       # ignores all the other information in the line, like charge, mass, etc. 

        elif parts[0] == "energy":
            energy = float(parts[1]) # extract the reference energy value

    if energy is None:
        raise ValueError("Energy missing in n2p2 block") # raising an error if energy is not found in the block

    if not symbols:
        raise ValueError("No atoms found in n2p2 block") # raising an error if no atoms are found in the block

    atoms = Atoms( #Converst extracted information into ASE Atoms object
        symbols=symbols,
        positions=positions,
        cell=cell if has_cell else None,
        pbc=has_cell
    )

    return atoms, energy

Number of structures: 51702
Reference energies: [-1412.31194807 -1415.37624938 -1413.92285234 ... -1438.08844783
 -1444.23706662 -1449.31426814]


A 'block' looks like this
 
```python
block = [
    "lattice 5.0 0.0 0.0",
    "lattice 0.0 5.0 0.0",
    "lattice 0.0 0.0 5.0",
    "atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0",
    "energy -10.5"
]
```

<img src="images/cell example.png" width="300">

In [ ]:
def read_n2p2(path):
    atoms_list = []
    energies = []
    block = []
    inside_block = False

    with open(path, "r") as file:
        for line in file:
            line = line.strip()

            if line == "begin":
                block = []
                inside_block = True

            elif line == "end":
                if not inside_block:
                    raise ValueError("Unexpected end")

                atoms, energy = _parse_n2p2_block(block)
                atoms_list.append(atoms)
                energies.append(energy)
                inside_block = False

            elif inside_block:
                block.append(line)

    if inside_block:
        raise ValueError("Unclosed n2p2 block")

    return atoms_list, np.array(energies)



# Example usage
if __name__ == "__main__":

    path = Path.cwd() / "input.data"

    atoms_list, energies = read_n2p2(path)

    print("Number of structures:", len(atoms_list))
    print("Reference energies:", energies)

TASK 2 ----------- Building descriptors

In [6]:

def build_descriptors(atoms_list):

    species = sorted({
        symbol
        for atoms in atoms_list
        for symbol in atoms.get_chemical_symbols()
    })

    acsf = ACSF(
        species=species,
        r_cut=6.0,
        g2_params=[
            [1.0, 0.0],
            [1.0, 2.0],
            [1.0, 4.0]
        ],
        g4_params=[
            [1.0, 1.0, 1.0],
            [1.0, 1.0, -1.0]
        ]
    )

    features = []

    for atoms in atoms_list:

        atomic_descriptors = acsf.create(atoms)

        structure_descriptor = np.mean(
            atomic_descriptors, axis=0
        )

        features.append(structure_descriptor)

    X = np.array(features)

    return X


# Example usage
if __name__ == "__main__":
    atoms_list, energies = read_n2p2("input.data")

    X = build_descriptors(atoms_list)

    print("Feature matrix shape:", X.shape)


Feature matrix shape: (51702, 6)
